## Simple attention mechanism

In [1]:
import torch

In [2]:
inputs =  torch.tensor([[0.43, 0.15, 0.89], # Your (x^1)
                        [0.55, 0.87, 0.66], # journey (x^2)
                        [0.57, 0.85, 0.64], # starts (x^3)
                        [0.22, 0.58, 0.33], # with (x^4)
                        [0.77, 0.25, 0.10], # one (x^5)
                        [0.05, 0.80, 0.55]] # step (x^6)
                        )

In [3]:
input_query = inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [4]:
input_1 = inputs[0]
input_1

tensor([0.4300, 0.1500, 0.8900])

In [5]:
torch.dot(input_query,input_1)

tensor(0.9544)

In [6]:
i  = 1
res = torch.dot(inputs[i],input_query)
res

tensor(1.4950)

In [7]:
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i,input_query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [8]:
attn_weights_2_temp = attn_scores_2/attn_scores_2.sum()
attn_weights_2_temp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [9]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [10]:
attn_weights_2 = torch.softmax(attn_scores_2,dim=0)

In [11]:
query = inputs[1]

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2+=attn_weights_2[i] * inputs[i]

context_vec_2


tensor([0.4419, 0.6515, 0.5683])

## 3.2.2. A simple self-attention mechanism wihtout trainable weights

In [12]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [13]:
inputs.T

tensor([[0.4300, 0.5500, 0.5700, 0.2200, 0.7700, 0.0500],
        [0.1500, 0.8700, 0.8500, 0.5800, 0.2500, 0.8000],
        [0.8900, 0.6600, 0.6400, 0.3300, 0.1000, 0.5500]])

In [14]:
attn_scores = torch.empty(6,6)

for i,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i,x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [15]:
attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [16]:
attn_weights = torch.softmax(attn_scores,dim=0)
attn_weights

tensor([[0.2098, 0.1385, 0.1390, 0.1435, 0.1526, 0.1385],
        [0.2006, 0.2379, 0.2369, 0.2074, 0.1958, 0.2184],
        [0.1981, 0.2333, 0.2326, 0.2046, 0.1975, 0.2128],
        [0.1242, 0.1240, 0.1242, 0.1462, 0.1367, 0.1420],
        [0.1220, 0.1082, 0.1108, 0.1263, 0.1879, 0.0988],
        [0.1452, 0.1581, 0.1565, 0.1720, 0.1295, 0.1896]])

In [17]:
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4017, 0.5023, 0.5059],
        [0.5595, 0.7824, 0.6953],
        [0.5538, 0.7686, 0.6834],
        [0.3369, 0.4647, 0.4119],
        [0.3525, 0.4059, 0.3657],
        [0.3856, 0.5761, 0.5077]])

## 3.4 Implementing self attention with trainable weights

### 3.4.1 Computing attention weights step by step

In [18]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [19]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

In [20]:
query_2 = x_2 @ W_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [21]:
keys = inputs @ W_key
value = inputs @ W_value

In [22]:
keys.shape

torch.Size([6, 2])

In [23]:
keys_2 = keys[1]
attn_score_22 = torch.dot(query_2,keys_2)
attn_score_22

tensor(1.8524, grad_fn=<DotBackward0>)

In [24]:
attn_scores_2 = query_2 @ keys.T
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [25]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2/d_k ** 0.5  ,dim=-1)

In [26]:
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [27]:
value

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]], grad_fn=<MmBackward0>)

In [28]:
context_vec = attn_weights_2 @ value
context_vec

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

## Implementing a compact Self attention mechanism

In [29]:
import torch.nn as nn

class SefAttention_v1(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))

    def forward(self,x):
        queries = inputs @ self.W_query
        keys = inputs @ self.W_key 
        values = inputs @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores/d_in**0.5,dim=-1)
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SefAttention_v1(d_in,d_out)
sa_v1(inputs)

tensor([[0.2961, 0.7970],
        [0.3016, 0.8104],
        [0.3013, 0.8098],
        [0.2921, 0.7873],
        [0.2904, 0.7833],
        [0.2956, 0.7959]], grad_fn=<MmBackward0>)

In [30]:
import torch.nn as nn

class SefAttention_v2(nn.Module):
    def __init__(self,d_in,d_out, qkv_bias = False ):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out, bias=qkv_bias)

    def forward(self,x):
        queries =  self.W_query(inputs)
        keys = self.W_key(inputs)
        values = self.W_value(inputs)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores/d_in**0.5,dim=-1)
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v2 = SefAttention_v2(d_in,d_out)
sa_v2(inputs)

tensor([[-0.5327, -0.1053],
        [-0.5315, -0.1076],
        [-0.5315, -0.1076],
        [-0.5294, -0.1073],
        [-0.5305, -0.1065],
        [-0.5295, -0.1077]], grad_fn=<MmBackward0>)

## 3.5 Hiding future words with causal masks
### Applying causal mask

In [31]:
queries =  sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores/d_in**0.5,dim=-1)

In [32]:
attn_weights

tensor([[0.1708, 0.1744, 0.1744, 0.1575, 0.1634, 0.1595],
        [0.1642, 0.1734, 0.1731, 0.1622, 0.1616, 0.1655],
        [0.1643, 0.1734, 0.1731, 0.1622, 0.1617, 0.1654],
        [0.1642, 0.1697, 0.1695, 0.1655, 0.1639, 0.1673],
        [0.1667, 0.1712, 0.1711, 0.1627, 0.1639, 0.1644],
        [0.1632, 0.1701, 0.1699, 0.1656, 0.1632, 0.1679]],
       grad_fn=<SoftmaxBackward0>)

In [33]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [34]:
masked_simple = attn_weights * mask_simple
masked_simple

tensor([[0.1708, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1642, 0.1734, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1643, 0.1734, 0.1731, 0.0000, 0.0000, 0.0000],
        [0.1642, 0.1697, 0.1695, 0.1655, 0.0000, 0.0000],
        [0.1667, 0.1712, 0.1711, 0.1627, 0.1639, 0.0000],
        [0.1632, 0.1701, 0.1699, 0.1656, 0.1632, 0.1679]],
       grad_fn=<MulBackward0>)

In [35]:
row_sums = mask_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = mask_simple/row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667]])


In [37]:
mask = torch.triu(torch.ones(context_length,context_length),diagonal=1)
masked = attn_scores.masked_fill(mask.bool(),-torch.inf)
masked

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)

In [39]:
attn_weights = torch.softmax(masked/keys.shape[-1]**0.5,dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


In [43]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)

In [44]:
example = torch.ones(6,6)
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [45]:
layer(example)

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])

In [46]:
layer(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.0000, 0.4936, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.4116, 0.3869, 0.3906, 0.0000],
        [0.3249, 0.3418, 0.0000, 0.0000, 0.3249, 0.3363]],
       grad_fn=<MulBackward0>)

### 3.5.3 Implementing a compact causal self - attention class

In [47]:
batch = torch.stack((inputs,inputs),dim=0)
batch.shape

torch.Size([2, 6, 3])

In [51]:
class CausalAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length, dropout, qkv_bias = False ):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out, bias=qkv_bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b, num_tokens, d_in = x.shape
        queries =  self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1 )
        attn_weights = self.dropout(attn_weights)

        context_vec  = attn_weights @ values


        return context_vec
    
torch.manual_seed(789)
dropout = 0
context_length = batch.shape[1]
ca = CausalAttention(d_in,d_out,context_length, dropout)
ca(batch)

tensor([[[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]],

        [[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]]], grad_fn=<UnsafeViewBackward0>)